# YNet trace-target sensitivity

The same simulated spectra are evaluated with three YNet seeds and L2 operators spanning trace targets 0.91 through 1.00. Results are saved as HDF5.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.trace_sweep.pipeline import run_evaluation

config_path = Path('../configs/TraceSweep/trace_sweep.yaml')
if not config_path.is_file():
    config_path = Path('configs/TraceSweep/trace_sweep.yaml')
config_path = config_path.resolve()
result_path = run_evaluation(config_path, overwrite=False)
result_path

In [ ]:
from walinet.paper_figures.trace_sweep.results import (
    aggregates_dataframe, results_dataframe,
)

individual = results_dataframe(result_path)
summary = aggregates_dataframe(result_path)
summary

In [ ]:
import matplotlib.pyplot as plt

subject_means = (
    individual.groupby(["subject", "target", "metric"], as_index=False)["mean"]
    .mean()
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
metric_labels = {
    "residual_l2_over_nuisance_l2": r"$\|\hat{n}-n_{\mathrm{true}}\|_2 / \|n_{\mathrm{true}}\|_2$",
    "residual_l2_over_clean_metabolite_l2": r"$\|\hat{n}-n_{\mathrm{true}}\|_2 / \|m_{\mathrm{clean}}\|_2$",
}
for axis, metric in zip(axes, metric_labels):
    for subject in subject_means["subject"].unique():
        rows = subject_means[(subject_means["subject"] == subject) &
                             (subject_means["metric"] == metric)].sort_values("target")
        axis.plot(rows["target"], rows["mean"], color="0.65",
                  linewidth=1, alpha=0.8)
    rows = summary[summary["metric"] == metric].sort_values("target")
    axis.errorbar(rows["target"], rows["mean"], yerr=rows["std"],
                  color="black", linewidth=2, marker="o", capsize=4,
                  label="mean ± SD across 5 subjects")
    axis.set_title(metric_labels[metric])
    axis.set_xlabel("L2 operator trace target")
    axis.set_ylabel("Relative L2 norm")
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()